# Foji Hygiene Recall Risk Prediction / Hygiene Recall Scheduling Recommendation

This notebook recreates the current **Foji hygiene recall** workflow in notebook form and makes one deliberate improvement:

> **The notebook ranks candidate hygiene slots by predicted probability of attendance rather than by a hard show / no-show class label.**

The notebook stays specific to Foji's use case:
- patient hygiene recall history from Open Dental-style appointment tables
- future open hygiene slots for a clinic day
- leakage-safe training rows from historical hygiene appointments
- ML scoring with `predict_proba(...)`
- ranking the top recall scheduling recommendations for a patient

## 1. Project Overview

Foji's hygiene recall scheduling problem can be framed as:

1. Build a patient attendance-history snapshot from prior hygiene appointments.
2. Generate candidate future hygiene slots for a clinic day.
3. Merge patient history features with slot features.
4. Score each patient-slot row with an attendance model.
5. Rank slots from highest to lowest predicted show probability.
6. Return the top recommended slots.

This notebook intentionally mirrors the current production-style Foji logic, but the recommender output is **probability-based** rather than **hard-label-based**.

## 2. Current Foji Workflow vs Notebook Workflow

### Current Foji production-style logic being replicated
- Build a patient hygiene-history summary over 12 months and 5 years.
- Generate open candidate slots for a clinic/day from future appointments.
- Create one row per patient-slot pair.
- Score rows with an attendance model.
- Apply operational tie-breakers such as adjacency and avoiding after-4pm slots.

### Important notebook improvement
The production-style workflow is often expressed as:
- predict a hard label (`show` / `no-show`)
- if any rows are predicted `show`, keep only those rows
- then rank them with tie-breakers

This notebook changes the **main recommender** to:
- predict a **show probability for every candidate slot**
- rank **all candidate slots** by `predicted_show_probability` descending
- use operational tie-breakers only as secondary sorting

That matters because recommendation quality depends more on **relative ordering** and **probability calibration** than on a single thresholded class boundary.

## 3. Data Inputs and Schema

This notebook uses realistic mock data to emulate the production sources.

### Source assumptions
- `od_patients`
- `od_appointments`
- `future_appointments`

### Appointment statuses used
- `Complete`
- `Broken`

### Training target
- `y_attend = 1` for `Complete`
- `y_attend = 0` for `Broken`

### Important notebook assumptions
- We use pandas DataFrames instead of ClickHouse / SQL queries.
- We assume `SecDateTEntry` is the scheduling timestamp for lead-time calculations when available.
- If true scheduling timestamps are missing, the notebook documents the fallback assumption.
- Adjacency features for future slots are computed from same-day booked appointments in an operatory.

In [ ]:
import math

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 140)

RANDOM_STATE = 42
TODAY = pd.Timestamp('2026-03-23 09:00:00')

## 4. Hygiene History Feature Construction

The function below reproduces the current Foji-style hygiene history snapshot in pandas.

### Required output fields
- `patient_id`
- `clinic_id`
- `patient_age`
- `show_count_12mo`
- `broken_count_12mo`
- `total_count_12mo`
- `p_base_smoothed`
- `days_since_last_completed`
- `days_since_last_broken`
- `broken_last_90d`
- `prior_appt_count_5y`
- `lifetime_show_rate_5y`
- `is_new_patient_5y`
- `scheduled_minutes`

The function returns a single-row DataFrame and includes the requested fallback defaults when no usable history exists.

In [ ]:
DEFAULT_HISTORY_VALUES = {
    'patient_age': -1,
    'show_count_12mo': 0,
    'broken_count_12mo': 0,
    'total_count_12mo': 0,
    'p_base_smoothed': 0.5,
    'days_since_last_completed': 9999,
    'days_since_last_broken': 9999,
    'broken_last_90d': 0,
    'prior_appt_count_5y': 0,
    'lifetime_show_rate_5y': 0.5,
    'is_new_patient_5y': 1,
    'scheduled_minutes': 60,
}


def calculate_age_years(birth_date: pd.Timestamp, reference_date: pd.Timestamp) -> int:
    if pd.isna(birth_date):
        return -1
    return int((reference_date.normalize() - birth_date.normalize()).days // 365.25)


def build_hygiene_history_snapshot(
    patient_id: int,
    clinic_id: int,
    scheduled_minutes: int,
    od_appointments: pd.DataFrame,
    od_patients: pd.DataFrame,
    reference_datetime: pd.Timestamp,
) -> pd.DataFrame:
    """Build a single-row hygiene history snapshot for a patient at scoring time."""
    base = {
        'patient_id': patient_id,
        'clinic_id': clinic_id,
        **DEFAULT_HISTORY_VALUES,
        'scheduled_minutes': scheduled_minutes,
    }

    patient_row = od_patients.loc[od_patients['PatNum'] == patient_id]
    if not patient_row.empty:
        birth_date = pd.to_datetime(patient_row.iloc[0]['birthDate'], errors='coerce')
        base['patient_age'] = calculate_age_years(birth_date, reference_datetime)

    appts = od_appointments.copy()
    appts['AptDateTime'] = pd.to_datetime(appts['AptDateTime'])
    appts = appts[
        (appts['PatNum'] == patient_id)
        & (appts['ClinicNum'] == clinic_id)
        & (appts['AptStatus'].isin(['Complete', 'Broken']))
        & (appts['AptDateTime'] < reference_datetime)
    ].sort_values('AptDateTime')

    if appts.empty:
        return pd.DataFrame([base])

    window_12mo_start = reference_datetime - pd.DateOffset(months=12)
    window_5y_start = reference_datetime - pd.DateOffset(years=5)
    window_90d_start = reference_datetime - pd.Timedelta(days=90)

    appts_12mo = appts[appts['AptDateTime'] >= window_12mo_start]
    appts_5y = appts[appts['AptDateTime'] >= window_5y_start]

    show_count_12mo = int((appts_12mo['AptStatus'] == 'Complete').sum())
    broken_count_12mo = int((appts_12mo['AptStatus'] == 'Broken').sum())
    total_count_12mo = show_count_12mo + broken_count_12mo

    completed_prior = appts[appts['AptStatus'] == 'Complete']
    broken_prior = appts[appts['AptStatus'] == 'Broken']
    completed_5y = appts_5y[appts_5y['AptStatus'] == 'Complete']

    last_completed = completed_prior['AptDateTime'].max() if not completed_prior.empty else pd.NaT
    last_broken = broken_prior['AptDateTime'].max() if not broken_prior.empty else pd.NaT

    prior_appt_count_5y = int(len(appts_5y))
    lifetime_show_rate_5y = (
        float(len(completed_5y) / prior_appt_count_5y) if prior_appt_count_5y > 0 else np.nan
    )

    base.update(
        {
            'show_count_12mo': show_count_12mo,
            'broken_count_12mo': broken_count_12mo,
            'total_count_12mo': total_count_12mo,
            'p_base_smoothed': (show_count_12mo + 2.0) / (show_count_12mo + broken_count_12mo + 4.0),
            'days_since_last_completed': 9999
            if pd.isna(last_completed)
            else int((reference_datetime.normalize() - last_completed.normalize()).days),
            'days_since_last_broken': 9999
            if pd.isna(last_broken)
            else int((reference_datetime.normalize() - last_broken.normalize()).days),
            'broken_last_90d': int(
                ((appts['AptStatus'] == 'Broken') & (appts['AptDateTime'] >= window_90d_start)).any()
            ),
            'prior_appt_count_5y': prior_appt_count_5y,
            'lifetime_show_rate_5y': lifetime_show_rate_5y if not np.isnan(lifetime_show_rate_5y) else 0.5,
            'is_new_patient_5y': int(prior_appt_count_5y < 3),
            'scheduled_minutes': scheduled_minutes,
        }
    )

    return pd.DataFrame([base])

### Mock Open Dental-style data

The notebook uses compact but realistic mock patient and appointment data so every section can run top-to-bottom.

In [ ]:
od_patients = pd.DataFrame(
    [
        {'PatNum': 1001, 'birthDate': '1985-04-10'},
        {'PatNum': 1002, 'birthDate': '1993-09-14'},
        {'PatNum': 1003, 'birthDate': '1978-01-05'},
        {'PatNum': 1004, 'birthDate': '2001-07-21'},
        {'PatNum': 1005, 'birthDate': '1969-11-30'},
        {'PatNum': 1006, 'birthDate': '1989-02-17'},
    ]
)


def make_mock_hygiene_appointments():
    rows = []
    apt_num = 1
    patient_patterns = {
        1001: ['Complete', 'Complete', 'Broken', 'Complete', 'Complete', 'Broken', 'Complete'],
        1002: ['Broken', 'Broken', 'Complete', 'Broken', 'Complete', 'Broken'],
        1003: ['Complete', 'Complete', 'Complete', 'Complete', 'Broken', 'Complete'],
        1004: ['Broken', 'Complete', 'Broken', 'Broken'],
        1005: ['Complete', 'Complete', 'Complete', 'Broken', 'Complete', 'Complete'],
        1006: ['Broken', 'Broken', 'Broken', 'Complete', 'Broken'],
    }
    patient_offsets = {
        1001: [900, 720, 500, 320, 170, 80, 25],
        1002: [880, 700, 550, 250, 120, 30],
        1003: [1200, 950, 730, 400, 145, 40],
        1004: [450, 260, 110, 15],
        1005: [1100, 840, 620, 300, 130, 10],
        1006: [760, 520, 210, 95, 35],
    }
    clinic_map = {1001: 1, 1002: 1, 1003: 1, 1004: 2, 1005: 2, 1006: 1}
    op_cycle = [1, 2, 3]
    provider_map = {1: 701, 2: 702, 3: 703}

    for pat_num, statuses in patient_patterns.items():
        clinic = clinic_map[pat_num]
        birth_date = pd.to_datetime(od_patients.loc[od_patients['PatNum'] == pat_num, 'birthDate'].iloc[0])
        for idx, status in enumerate(statuses):
            apt_dt = TODAY - pd.Timedelta(days=patient_offsets[pat_num][idx]) + pd.Timedelta(hours=(idx % 6) + 8)
            scheduled_minutes = 60 if idx % 3 != 0 else 90
            end_dt = apt_dt + pd.Timedelta(minutes=scheduled_minutes)
            sec_entry = apt_dt - pd.Timedelta(days=max(7, 20 + (idx * 9) - (5 if status == 'Broken' else 0)))
            op = op_cycle[(idx + pat_num) % len(op_cycle)]
            prov_num = provider_map[op]
            prov_hyg = prov_num if idx % 2 == 0 else 0
            rows.append(
                {
                    'AptNum': apt_num,
                    'PatNum': pat_num,
                    'ClinicNum': clinic,
                    'Op': op,
                    'ProvNum': prov_num,
                    'ProvHyg': prov_hyg,
                    'AptStatus': status,
                    'IsHygiene': 1,
                    'AptDateTime': apt_dt,
                    'endTime': end_dt,
                    'SecDateTEntry': sec_entry,
                    'Pattern': 'X' * (scheduled_minutes // 5),
                    'birthDate': birth_date,
                }
            )
            apt_num += 1
    return pd.DataFrame(rows).sort_values(['PatNum', 'AptDateTime']).reset_index(drop=True)


od_appointments = make_mock_hygiene_appointments()

future_appointments = pd.DataFrame(
    [
        {'ClinicNum': 1, 'OperatoryNum': 1, 'AptDateTime': '2026-04-15 08:00', 'endTime': '2026-04-15 09:00'},
        {'ClinicNum': 1, 'OperatoryNum': 1, 'AptDateTime': '2026-04-15 10:00', 'endTime': '2026-04-15 11:00'},
        {'ClinicNum': 1, 'OperatoryNum': 1, 'AptDateTime': '2026-04-15 14:00', 'endTime': '2026-04-15 15:00'},
        {'ClinicNum': 1, 'OperatoryNum': 2, 'AptDateTime': '2026-04-15 09:00', 'endTime': '2026-04-15 10:00'},
        {'ClinicNum': 1, 'OperatoryNum': 2, 'AptDateTime': '2026-04-15 13:00', 'endTime': '2026-04-15 14:00'},
        {'ClinicNum': 1, 'OperatoryNum': 2, 'AptDateTime': '2026-04-15 15:00', 'endTime': '2026-04-15 16:00'},
        {'ClinicNum': 1, 'OperatoryNum': 3, 'AptDateTime': '2026-04-15 08:00', 'endTime': '2026-04-15 10:00'},
        {'ClinicNum': 1, 'OperatoryNum': 3, 'AptDateTime': '2026-04-15 11:00', 'endTime': '2026-04-15 12:00'},
        {'ClinicNum': 1, 'OperatoryNum': 3, 'AptDateTime': '2026-04-15 16:00', 'endTime': '2026-04-15 17:00'},
    ]
)
future_appointments['AptDateTime'] = pd.to_datetime(future_appointments['AptDateTime'])
future_appointments['endTime'] = pd.to_datetime(future_appointments['endTime'])

print('od_patients')
display(od_patients)
print('od_appointments sample')
display(od_appointments.head(10))
print('future_appointments')
display(future_appointments)

In [ ]:
example_snapshot = build_hygiene_history_snapshot(
    patient_id=1001,
    clinic_id=1,
    scheduled_minutes=60,
    od_appointments=od_appointments,
    od_patients=od_patients,
    reference_datetime=TODAY,
)

example_snapshot

## 5. Future Slot Construction

The next function recreates the Foji-style open-slot generation process for a target clinic day:

1. Identify operatories with appointments that day.
2. Collect booked ranges by operatory.
3. Generate candidate slot starts every 60 minutes.
4. Build slots with the requested `scheduled_minutes`.
5. Remove overlapping slots.
6. Compute slot-level time and adjacency features.

In [ ]:
def generate_slot_time_features(slot_start: pd.Timestamp, scheduled_minutes: int, score_generated_at: pd.Timestamp) -> dict:
    lead_time_days = max(int((slot_start.normalize() - score_generated_at.normalize()).days), 0)
    month_of_year = int(slot_start.month)
    month_angle = 2 * math.pi * month_of_year / 12.0
    return {
        'scheduled_minutes': scheduled_minutes,
        'dow_md': int(slot_start.dayofweek),
        'hour_of_day': int(slot_start.hour),
        'is_morning': int(slot_start.hour < 12),
        'is_after_4pm': int(slot_start.hour >= 16),
        'is_monday_or_friday': int(slot_start.dayofweek in [0, 4]),
        'lead_time_days': lead_time_days,
        'lead_0_7': int(lead_time_days <= 7),
        'lead_8_30': int(8 <= lead_time_days <= 30),
        'lead_31_90': int(31 <= lead_time_days <= 90),
        'lead_90_plus': int(lead_time_days > 90),
        'month_of_year': month_of_year,
        'month_sin': float(np.sin(month_angle)),
        'month_cos': float(np.cos(month_angle)),
    }


def generate_open_slots(
    clinic_id: int,
    target_month: int,
    target_day: int,
    scheduled_minutes: int,
    future_appointments: pd.DataFrame,
    score_generated_at: pd.Timestamp,
    year: int = 2026,
    day_start_hour: int = 8,
    day_end_hour: int = 18,
    step_min: int = 60,
) -> pd.DataFrame:
    target_date = pd.Timestamp(year=year, month=target_month, day=target_day)
    day_start = target_date + pd.Timedelta(hours=day_start_hour)
    day_end = target_date + pd.Timedelta(hours=day_end_hour)

    appts = future_appointments.copy()
    appts['AptDateTime'] = pd.to_datetime(appts['AptDateTime'])
    appts['endTime'] = pd.to_datetime(appts['endTime'])
    day_appts = appts[
        (appts['ClinicNum'] == clinic_id)
        & (appts['AptDateTime'].dt.normalize() == target_date.normalize())
    ].copy()

    if day_appts.empty:
        return pd.DataFrame(columns=[
            'ClinicNum', 'OperatoryNum', 'slot_start', 'slot_end', 'scheduled_minutes', 'dow_md', 'hour_of_day',
            'is_morning', 'is_after_4pm', 'is_monday_or_friday', 'lead_time_days', 'lead_0_7', 'lead_8_30',
            'lead_31_90', 'lead_90_plus', 'month_of_year', 'month_sin', 'month_cos', 'touches_existing_appt',
            'is_after_existing_appt', 'is_before_existing_appt'
        ])

    operatories = sorted(day_appts['OperatoryNum'].unique())
    candidate_starts = pd.date_range(day_start, day_end - pd.Timedelta(minutes=scheduled_minutes), freq=f'{step_min}min')

    rows = []
    for operatory in operatories:
        op_appts = day_appts[day_appts['OperatoryNum'] == operatory].sort_values('AptDateTime')
        booked_ranges = list(zip(op_appts['AptDateTime'], op_appts['endTime']))

        for slot_start in candidate_starts:
            slot_end = slot_start + pd.Timedelta(minutes=scheduled_minutes)
            overlaps = any((slot_start < booked_end) and (slot_end > booked_start) for booked_start, booked_end in booked_ranges)
            if overlaps:
                continue

            is_after_existing_appt = int(any(booked_end == slot_start for _, booked_end in booked_ranges))
            is_before_existing_appt = int(any(booked_start == slot_end for booked_start, _ in booked_ranges))
            touches_existing_appt = int(is_after_existing_appt or is_before_existing_appt)

            row = {
                'ClinicNum': clinic_id,
                'OperatoryNum': operatory,
                'slot_start': slot_start,
                'slot_end': slot_end,
                'touches_existing_appt': touches_existing_appt,
                'is_after_existing_appt': is_after_existing_appt,
                'is_before_existing_appt': is_before_existing_appt,
            }
            row.update(generate_slot_time_features(slot_start, scheduled_minutes, score_generated_at))
            rows.append(row)

    return pd.DataFrame(rows).sort_values(['OperatoryNum', 'slot_start']).reset_index(drop=True)

In [ ]:
open_slots = generate_open_slots(
    clinic_id=1,
    target_month=4,
    target_day=15,
    scheduled_minutes=60,
    future_appointments=future_appointments,
    score_generated_at=TODAY,
)

open_slots.head(20)

## 6. Merge Logic for Patient + Slot Features

We now create one row per patient-slot pair by cross joining the single-row patient snapshot with the candidate slots.

In [ ]:
PATIENT_HISTORY_FEATURES = [
    'broken_count_12mo',
    'broken_last_90d',
    'days_since_last_broken',
    'days_since_last_completed',
    'lifetime_show_rate_5y',
    'p_base_smoothed',
    'patient_age',
    'prior_appt_count_5y',
    'show_count_12mo',
    'total_count_12mo',
    'is_new_patient_5y',
]

SLOT_FEATURES = [
    'dow_md',
    'hour_of_day',
    'is_after_4pm',
    'is_monday_or_friday',
    'is_morning',
    'lead_0_7',
    'lead_8_30',
    'lead_31_90',
    'lead_90_plus',
    'lead_time_days',
    'month_of_year',
    'month_sin',
    'month_cos',
    'scheduled_minutes',
    'touches_existing_appt',
    'is_after_existing_appt',
    'is_before_existing_appt',
]

METADATA_COLUMNS = ['slot_start', 'slot_end', 'ClinicNum', 'OperatoryNum']


def merge_patient_history_with_slots(patient_history_snapshot: pd.DataFrame, candidate_slots: pd.DataFrame) -> pd.DataFrame:
    patient = patient_history_snapshot.copy()
    slots = candidate_slots.copy()
    patient['merge_key'] = 1
    slots['merge_key'] = 1
    merged = patient.merge(slots, on='merge_key', how='inner').drop(columns=['merge_key'])
    merged = merged.rename(columns={'clinic_id': 'patient_clinic_id'})
    ordered_columns = [
        'patient_id',
        'patient_clinic_id',
        *PATIENT_HISTORY_FEATURES,
        *SLOT_FEATURES,
        *METADATA_COLUMNS,
    ]
    return merged[ordered_columns]

In [ ]:
merged_patient_slots = merge_patient_history_with_slots(example_snapshot, open_slots)
merged_patient_slots.head(10)

## 7. Training Dataset Design

This section shows how to build **historical training rows** from past hygiene appointments.

### Training design rules
- One row per historical hygiene appointment.
- Use only feature information available **before** the appointment being predicted.
- Compute patient history leakage-safely from prior appointments.
- Create timing features from the appointment slot itself.
- Derive `lead_time_days` from `SecDateTEntry` when available.
- If true scheduling timestamp is unavailable, document the assumption or fallback.

### Target
- `y_attend = 1` if `AptStatus == 'Complete'`
- `y_attend = 0` if `AptStatus == 'Broken'`

In [ ]:
def compute_history_features_prior_to_index(
    appointment_row: pd.Series,
    prior_patient_appts: pd.DataFrame,
    birth_date: pd.Timestamp,
) -> dict:
    reference_datetime = pd.to_datetime(appointment_row['AptDateTime'])
    appts = prior_patient_appts.copy()
    appts['AptDateTime'] = pd.to_datetime(appts['AptDateTime'])
    appts = appts[
        (appts['AptStatus'].isin(['Complete', 'Broken']))
        & (appts['AptDateTime'] < reference_datetime)
    ].sort_values('AptDateTime')

    window_12mo_start = reference_datetime - pd.DateOffset(months=12)
    window_5y_start = reference_datetime - pd.DateOffset(years=5)
    window_90d_start = reference_datetime - pd.Timedelta(days=90)

    appts_12mo = appts[appts['AptDateTime'] >= window_12mo_start]
    appts_5y = appts[appts['AptDateTime'] >= window_5y_start]
    complete_12mo = appts_12mo[appts_12mo['AptStatus'] == 'Complete']
    broken_12mo = appts_12mo[appts_12mo['AptStatus'] == 'Broken']
    complete_5y = appts_5y[appts_5y['AptStatus'] == 'Complete']

    show_count_12mo = int(len(complete_12mo))
    broken_count_12mo = int(len(broken_12mo))
    total_count_12mo = show_count_12mo + broken_count_12mo
    prior_appt_count_5y = int(len(appts_5y))

    last_completed = complete_12mo['AptDateTime'].max() if not complete_12mo.empty else (
        appts[appts['AptStatus'] == 'Complete']['AptDateTime'].max() if not appts.empty else pd.NaT
    )
    last_broken = broken_12mo['AptDateTime'].max() if not broken_12mo.empty else (
        appts[appts['AptStatus'] == 'Broken']['AptDateTime'].max() if not appts.empty else pd.NaT
    )

    return {
        'patient_age': calculate_age_years(pd.to_datetime(birth_date), reference_datetime),
        'show_count_12mo': show_count_12mo,
        'broken_count_12mo': broken_count_12mo,
        'total_count_12mo': total_count_12mo,
        'p_base_smoothed': (show_count_12mo + 2.0) / (show_count_12mo + broken_count_12mo + 4.0),
        'days_since_last_completed': 9999 if pd.isna(last_completed) else int((reference_datetime.normalize() - last_completed.normalize()).days),
        'days_since_last_broken': 9999 if pd.isna(last_broken) else int((reference_datetime.normalize() - last_broken.normalize()).days),
        'broken_last_90d': int(((appts['AptStatus'] == 'Broken') & (appts['AptDateTime'] >= window_90d_start)).any()),
        'prior_appt_count_5y': prior_appt_count_5y,
        'lifetime_show_rate_5y': float(len(complete_5y) / prior_appt_count_5y) if prior_appt_count_5y > 0 else 0.5,
        'is_new_patient_5y': int(prior_appt_count_5y < 3),
    }


def build_training_rows(appointments: pd.DataFrame) -> pd.DataFrame:
    appts = appointments.copy()
    appts['AptDateTime'] = pd.to_datetime(appts['AptDateTime'])
    appts['endTime'] = pd.to_datetime(appts['endTime'])
    appts['SecDateTEntry'] = pd.to_datetime(appts['SecDateTEntry'], errors='coerce')
    appts = appts[
        (appts['IsHygiene'] == 1)
        & (appts['AptStatus'].isin(['Complete', 'Broken']))
    ].sort_values(['PatNum', 'AptDateTime']).reset_index(drop=True)

    rows = []
    for _, row in appts.iterrows():
        prior_patient_appts = appts[appts['PatNum'] == row['PatNum']]
        history = compute_history_features_prior_to_index(row, prior_patient_appts, row['birthDate'])

        scheduled_minutes = int(len(row['Pattern']) * 5) if pd.notna(row['Pattern']) else int((row['endTime'] - row['AptDateTime']).total_seconds() // 60)
        slot_features = generate_slot_time_features(
            slot_start=pd.to_datetime(row['AptDateTime']),
            scheduled_minutes=scheduled_minutes,
            score_generated_at=pd.to_datetime(row['SecDateTEntry']) if pd.notna(row['SecDateTEntry']) else pd.to_datetime(row['AptDateTime']) - pd.Timedelta(days=30),
        )

        training_row = {
            'AptNum': row['AptNum'],
            'PatNum': row['PatNum'],
            'ClinicNum': row['ClinicNum'],
            'OperatoryNum': row['Op'],
            'HygProvNum': row['ProvHyg'] if row['ProvHyg'] not in [0, None, np.nan] else row['ProvNum'],
            **history,
            **slot_features,
            'touches_existing_appt': 0,
            'is_after_existing_appt': 0,
            'is_before_existing_appt': 0,
            'y_attend': int(row['AptStatus'] == 'Complete'),
        }
        rows.append(training_row)

    return pd.DataFrame(rows)

In [ ]:
training_rows = build_training_rows(od_appointments)
print(training_rows.shape)
display(training_rows.head(12))

## 8. Model Training

We train two sklearn models:

1. **Logistic Regression** baseline
2. **Random Forest** tree-based model

### Why probability quality matters here
For slot recommendation, it is usually better to know:
- slot A has a 0.78 predicted show probability
- slot B has a 0.73 predicted show probability
- slot C has a 0.56 predicted show probability

than to know only whether each slot crosses a threshold like 0.50.

A threshold can be useful for reporting, but the **ranking should be based on probability**. That is why the notebook uses `predict_proba(X)[:, 1]` for recommendation.

In [ ]:
MODEL_NUMERIC_FEATURES = [
    'patient_age',
    'show_count_12mo',
    'broken_count_12mo',
    'total_count_12mo',
    'p_base_smoothed',
    'days_since_last_completed',
    'days_since_last_broken',
    'broken_last_90d',
    'prior_appt_count_5y',
    'lifetime_show_rate_5y',
    'is_new_patient_5y',
    'scheduled_minutes',
    'dow_md',
    'hour_of_day',
    'is_morning',
    'is_after_4pm',
    'is_monday_or_friday',
    'lead_time_days',
    'lead_0_7',
    'lead_8_30',
    'lead_31_90',
    'lead_90_plus',
    'month_of_year',
    'month_sin',
    'month_cos',
    'touches_existing_appt',
    'is_after_existing_appt',
    'is_before_existing_appt',
]
MODEL_CATEGORICAL_FEATURES = ['ClinicNum', 'OperatoryNum', 'HygProvNum']
MODEL_FEATURES = MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES


def make_model_pipeline(model_name: str = 'logistic') -> Pipeline:
    numeric_pipeline = Pipeline(
        steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]
    )
    categorical_pipeline = Pipeline(
        steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore')),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_pipeline, MODEL_NUMERIC_FEATURES),
            ('cat', categorical_pipeline, MODEL_CATEGORICAL_FEATURES),
        ]
    )

    if model_name == 'logistic':
        estimator = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
    elif model_name == 'random_forest':
        estimator = RandomForestClassifier(
            n_estimators=300,
            max_depth=6,
            min_samples_leaf=2,
            random_state=RANDOM_STATE,
        )
    else:
        raise ValueError(f'Unsupported model_name: {model_name}')

    return Pipeline(steps=[('preprocessor', preprocessor), ('model', estimator)])


def evaluate_classifier(model: Pipeline, X_test: pd.DataFrame, y_test: pd.Series, threshold: float = 0.5) -> dict:
    pred_proba = model.predict_proba(X_test)[:, 1]
    pred_label = (pred_proba >= threshold).astype(int)
    return {
        'roc_auc': roc_auc_score(y_test, pred_proba),
        'average_precision': average_precision_score(y_test, pred_proba),
        'log_loss': log_loss(y_test, pred_proba, labels=[0, 1]),
        'accuracy': accuracy_score(y_test, pred_label),
        'precision': precision_score(y_test, pred_label, zero_division=0),
        'recall': recall_score(y_test, pred_label, zero_division=0),
        'f1': f1_score(y_test, pred_label, zero_division=0),
        'brier_score': brier_score_loss(y_test, pred_proba),
        'confusion_matrix': confusion_matrix(y_test, pred_label),
    }

In [ ]:
X = training_rows[MODEL_FEATURES]
y = training_rows['y_attend']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.35,
    random_state=RANDOM_STATE,
    stratify=y,
)

logistic_pipeline = make_model_pipeline('logistic')
rf_pipeline = make_model_pipeline('random_forest')

logistic_pipeline.fit(X_train, y_train)
rf_pipeline.fit(X_train, y_train)

logistic_metrics = evaluate_classifier(logistic_pipeline, X_test, y_test)
rf_metrics = evaluate_classifier(rf_pipeline, X_test, y_test)

metrics_df = pd.DataFrame([
    {'model': 'LogisticRegression', **{k: v for k, v in logistic_metrics.items() if k != 'confusion_matrix'}},
    {'model': 'RandomForestClassifier', **{k: v for k, v in rf_metrics.items() if k != 'confusion_matrix'}},
]).set_index('model')

print('Model metrics')
display(metrics_df.round(4))
print('Logistic confusion matrix')
display(pd.DataFrame(logistic_metrics['confusion_matrix'], index=['actual_0', 'actual_1'], columns=['pred_0', 'pred_1']))
print('Random forest confusion matrix')
display(pd.DataFrame(rf_metrics['confusion_matrix'], index=['actual_0', 'actual_1'], columns=['pred_0', 'pred_1']))

### Calibration commentary

For recall scheduling recommendations, **probability calibration** matters because the model is being used to **rank alternatives**. A model that outputs good probabilities can support:
- better slot ranking
- better threshold setting for outreach workflows
- more interpretable trade-offs between operational preferences and attendance likelihood

This notebook reports **log loss** and **Brier score** in addition to thresholded metrics. In a production version, Foji should also evaluate:
- calibration plots / reliability curves
- Platt scaling or isotonic calibration
- temporal validation by booking period or clinic rollout date

## 9. Probability-Based Slot Scoring

The scoring function below applies a trained model to every merged patient-slot row and stores:
- `predicted_show_probability`
- optional thresholded `predicted_show_label`

The key design rule is that **every slot receives a probability**, even if it would not cross the binary threshold.

In [ ]:
SCORING_OUTPUT_COLUMNS = [
    'slot_start',
    'slot_end',
    'ClinicNum',
    'OperatoryNum',
    'predicted_show_probability',
    'predicted_show_label',
    'touches_existing_appt',
    'is_after_existing_appt',
    'is_before_existing_appt',
    'is_after_4pm',
]


def prepare_scoring_features(merged_rows: pd.DataFrame) -> pd.DataFrame:
    scoring = merged_rows.copy()
    scoring['ClinicNum'] = scoring['ClinicNum'].astype(int)
    scoring['OperatoryNum'] = scoring['OperatoryNum'].astype(int)
    if 'HygProvNum' not in scoring.columns:
        scoring['HygProvNum'] = 701
    return scoring


def score_slots(merged_rows: pd.DataFrame, trained_model: Pipeline, threshold: float = 0.5) -> pd.DataFrame:
    scoring_rows = prepare_scoring_features(merged_rows)
    proba = trained_model.predict_proba(scoring_rows[MODEL_FEATURES])[:, 1]
    scored = scoring_rows.copy()
    scored['predicted_show_probability'] = proba
    scored['predicted_show_label'] = (scored['predicted_show_probability'] >= threshold).astype(int)
    return scored[SCORING_OUTPUT_COLUMNS].sort_values('predicted_show_probability', ascending=False).reset_index(drop=True)

## 10. Slot Ranking and Recommendation

### Notebook recommendation logic (improved)
Primary ranking:
1. `predicted_show_probability` descending
2. `touches_existing_appt` descending
3. `is_after_4pm` ascending
4. `slot_start` ascending

### Current Foji production-style comparison logic
- filter to `predicted_show_label == 1` if any rows exist
- otherwise use all rows
- then sort by adjacency, after-4pm avoidance, earliest slot

The notebook uses the **probability-based ranking** as the main recommendation output.

In [ ]:
def select_best_slots(scored_slots: pd.DataFrame, top_n: int = 4) -> pd.DataFrame:
    ranked = scored_slots.sort_values(
        by=['predicted_show_probability', 'touches_existing_appt', 'is_after_4pm', 'slot_start'],
        ascending=[False, False, True, True],
    ).reset_index(drop=True)
    ranked['recommendation_rank'] = np.arange(1, len(ranked) + 1)
    return ranked.head(top_n)


def select_best_slots_current_logic(scored_slots: pd.DataFrame, top_n: int = 4) -> pd.DataFrame:
    candidates = scored_slots.copy()
    if (candidates['predicted_show_label'] == 1).any():
        candidates = candidates[candidates['predicted_show_label'] == 1].copy()
    ranked = candidates.sort_values(
        by=['touches_existing_appt', 'is_after_4pm', 'slot_start'],
        ascending=[False, True, True],
    ).reset_index(drop=True)
    ranked['current_logic_rank'] = np.arange(1, len(ranked) + 1)
    return ranked.head(top_n)

## 11. Comparison to Current Foji Rule-Based Logic

The comparison below highlights the difference between:
- **probability-first ranking** used by this notebook, and
- **threshold-first ranking** representing the current production-style logic.

This is important because threshold-first logic can discard useful ordering information among viable slots.

In [ ]:
trained_model = rf_pipeline
scored_slots = score_slots(merged_patient_slots, trained_model=trained_model, threshold=0.5)
notebook_recommendations = select_best_slots(scored_slots, top_n=4)
current_logic_recommendations = select_best_slots_current_logic(scored_slots, top_n=4)

print('All scored slots')
display(scored_slots.head(12))
print('Notebook probability-based recommendations')
display(notebook_recommendations)
print('Current Foji-style comparison recommendations')
display(current_logic_recommendations)

## 12. Example End-to-End Recommendation

This section runs the whole notebook workflow end-to-end using a mock recall patient:

1. build patient history snapshot
2. generate open slots for a future day
3. merge patient history with slots
4. score all slots with predicted probabilities
5. rank the slots
6. return the top 4 recommendations

In [ ]:
TARGET_PATIENT_ID = 1002
TARGET_CLINIC_ID = 1
TARGET_SCHEDULED_MINUTES = 60
TARGET_MONTH = 4
TARGET_DAY = 15

patient_snapshot = build_hygiene_history_snapshot(
    patient_id=TARGET_PATIENT_ID,
    clinic_id=TARGET_CLINIC_ID,
    scheduled_minutes=TARGET_SCHEDULED_MINUTES,
    od_appointments=od_appointments,
    od_patients=od_patients,
    reference_datetime=TODAY,
)

candidate_slots = generate_open_slots(
    clinic_id=TARGET_CLINIC_ID,
    target_month=TARGET_MONTH,
    target_day=TARGET_DAY,
    scheduled_minutes=TARGET_SCHEDULED_MINUTES,
    future_appointments=future_appointments,
    score_generated_at=TODAY,
)

patient_slot_rows = merge_patient_history_with_slots(patient_snapshot, candidate_slots)
patient_slot_rows['HygProvNum'] = 701

scored_recall_slots = score_slots(patient_slot_rows, trained_model=trained_model, threshold=0.5)
top_recommendations = select_best_slots(scored_recall_slots, top_n=4)

final_output = top_recommendations[
    [
        'slot_start',
        'slot_end',
        'OperatoryNum',
        'predicted_show_probability',
        'predicted_show_label',
        'touches_existing_appt',
        'is_after_existing_appt',
        'is_before_existing_appt',
        'is_after_4pm',
        'recommendation_rank',
    ]
].copy()
final_output['predicted_show_probability'] = final_output['predicted_show_probability'].round(4)

print('Patient snapshot')
display(patient_snapshot)
print('Candidate slots')
display(candidate_slots.head(12))
print('Final top-4 recommendations')
display(final_output)

## 13. Limitations and Next Steps

### What parts of the current Foji workflow were replicated
- Patient hygiene history snapshot over 12 months and 5 years.
- Future open-slot generation by clinic day and operatory.
- Patient-slot row construction.
- Historical training row design using prior-only patient history.
- Attendance model scoring.
- Current-style adjacency / after-4pm / earliest tie-break logic.

### What was intentionally improved for the notebook
- The notebook recommender is **probability-based**.
- Every candidate slot receives `predicted_show_probability`.
- Ranking is based primarily on that probability rather than on a thresholded label.

### Why probability-based ranking is better for this use case
- Scheduling recommendation is a ranking problem before it is a thresholding problem.
- A calibrated probability preserves the strength of preference among candidate slots.
- Thresholding can collapse several meaningfully different slots into the same hard class.
- Clinics can later tune thresholds for operational workflows without changing the ranking model.

### Notebook assumptions that would need production replacement
- Mock pandas DataFrames replace Open Dental / ClickHouse data pulls.
- The slot generator uses a simplified same-day operatory schedule.
- Historical adjacency features are held at zero in training because the mock training data does not reconstruct surrounding bookings.
- `SecDateTEntry` is assumed to be the scheduling timestamp for `lead_time_days`; if unavailable, the notebook uses a documented fallback assumption.

### Production data replacements needed
- Replace mock patient data with real `od_patients` extracts.
- Replace mock appointment history with real hygiene appointment facts from Open Dental / ClickHouse.
- Replace future slot mocks with real schedule availability logic.
- Recompute operational adjacency features from actual booked schedules.
- Use real provider / clinic / operatory identifiers and validate category cardinality.

### Important future improvements
- Evaluate probability calibration more carefully.
- Consider calibrated classifiers.
- Compare operational tie-breakers versus learned ranking directly.
- Confirm whether adjacency features should be model inputs, tie-breakers, or both.
- Validate leakage-safe availability of lead-time features in real historical training data.